## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [5]:
df1 = pl.read_csv("data/test_noLabel.csv", encoding="utf8-lossy")
df2 = pl.read_csv("data/submit_example.csv", encoding="utf8-lossy")
df3 = pl.read_csv("data/train.csv", encoding="utf8-lossy")

# Join df1 and df2 on the "ID" column (you can adjust join type if needed)
joined_df = df1.join(df2, on="ID", how="inner")

# Concatenate the joined result with df3 (assumes matching schema)
df = pl.concat([joined_df, df3], how="vertical", rechunk=True)

df

ID,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Label
i64,i64,str,str,i64,i64,str,i64,i64,str,i64,i64,str,i64,str,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1100,40,"""Non-Travel""","""Research & Development""",9,4,"""Other""",1449,3,"""Male""",3,2,"""Laboratory Technician""",3,"""Divorced""",3975,3,"""Y""","""No""",11,3,3,80,2,11,2,4,8,7,0,7,0
1101,53,"""Travel_Rarely""","""Research & Development""",7,2,"""Medical""",1201,4,"""Female""",3,5,"""Manager""",3,"""Divorced""",18606,3,"""Y""","""No""",18,3,2,80,1,26,6,3,7,7,4,7,0
1102,42,"""Travel_Rarely""","""Research & Development""",2,4,"""Other""",477,1,"""Male""",2,2,"""Healthcare Representative""",4,"""Single""",6781,3,"""Y""","""No""",23,4,2,80,0,14,6,3,1,0,0,0,0
1103,34,"""Travel_Frequently""","""Human Resources""",11,3,"""Life Sciences""",1289,3,"""Male""",2,2,"""Human Resources""",2,"""Married""",4490,4,"""Y""","""No""",11,3,4,80,2,14,5,4,10,9,1,8,0
1104,32,"""Travel_Rarely""","""Research & Development""",1,1,"""Life Sciences""",134,4,"""Male""",3,1,"""Research Scientist""",1,"""Single""",2956,1,"""Y""","""No""",13,3,4,80,0,1,2,3,1,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1095,35,"""Travel_Rarely""","""Research & Development""",23,4,"""Medical""",75,3,"""Female""",3,1,"""Laboratory Technician""",1,"""Married""",4014,3,"""Y""","""Yes""",15,3,3,80,1,4,3,3,2,2,2,2,0
1096,38,"""Travel_Rarely""","""Sales""",2,4,"""Marketing""",1835,2,"""Female""",1,2,"""Sales Representative""",4,"""Married""",5405,2,"""Y""","""Yes""",20,4,1,80,2,20,4,2,4,2,0,3,0
1097,37,"""Travel_Rarely""","""Sales""",16,4,"""Marketing""",868,4,"""Male""",2,2,"""Sales Executive""",3,"""Divorced""",6334,4,"""Y""","""No""",19,3,4,80,2,9,2,3,1,0,0,0,0


### Retrieve Number of Nulls in Each Feature

In [6]:
def count_nulls(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "feature": df.columns,
        "null_count": df.null_count().row(0)
    })

pl.Config.set_tbl_rows(35)

null_counts = count_nulls(df)
null_counts

feature,null_count
str,i64
"""ID""",0
"""Age""",0
"""BusinessTravel""",0
"""Department""",0
"""DistanceFromHome""",0
"""Education""",0
"""EducationField""",0
"""EmployeeNumber""",0
"""EnvironmentSatisfaction""",0


### Retrieve Basic Information About DataFrame

In [7]:
def print_schema(df: pl.DataFrame):
    print(f"{'Column':<30} | {'Data Type'}")
    print("-" * 60)
    for name, dtype in zip(df.columns, df.dtypes):
        print(f"{name:<30} | {dtype}")

print_schema(df)

Column                         | Data Type
------------------------------------------------------------
ID                             | Int64
Age                            | Int64
BusinessTravel                 | String
Department                     | String
DistanceFromHome               | Int64
Education                      | Int64
EducationField                 | String
EmployeeNumber                 | Int64
EnvironmentSatisfaction        | Int64
Gender                         | String
JobInvolvement                 | Int64
JobLevel                       | Int64
JobRole                        | String
JobSatisfaction                | Int64
MaritalStatus                  | String
MonthlyIncome                  | Int64
NumCompaniesWorked             | Int64
Over18                         | String
OverTime                       | String
PercentSalaryHike              | Int64
PerformanceRating              | Int64
RelationshipSatisfaction       | Int64
StandardHours                 

### Display Summary Statistics for All Columns

In [8]:
summary = df.describe()
print(summary)

shape: (9, 33)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ ID        ┆ Age       ┆ BusinessT ┆ … ┆ YearsInCu ┆ YearsSinc ┆ YearsWith ┆ Label    │
│ ---       ┆ ---       ┆ ---       ┆ ravel     ┆   ┆ rrentRole ┆ eLastProm ┆ CurrManag ┆ ---      │
│ str       ┆ f64       ┆ f64       ┆ ---       ┆   ┆ ---       ┆ otion     ┆ er        ┆ f64      │
│           ┆           ┆           ┆ str       ┆   ┆ f64       ┆ ---       ┆ ---       ┆          │
│           ┆           ┆           ┆           ┆   ┆           ┆ f64       ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 1450.0    ┆ 1450.0    ┆ 1450      ┆ … ┆ 1450.0    ┆ 1450.0    ┆ 1450.0    ┆ 1450.0   │
│ null_coun ┆ 0.0       ┆ 0.0       ┆ 0         ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           

### Find Longest Text Length in Each Column

In [9]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

BusinessTravel,Department,EducationField,Gender,JobRole,MaritalStatus,Over18,OverTime
u32,u32,u32,u32,u32,u32,u32,u32
17,22,16,6,25,8,1,3


### Retrieve Data Types of All Columns

In [10]:
print("Column data types:\n", df.dtypes)

Column data types:
 [Int64, Int64, String, String, Int64, Int64, String, Int64, Int64, String, Int64, Int64, String, Int64, String, Int64, Int64, String, String, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64]


### Count Unique Values in Each Column

In [11]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                           Unique values in ID : 1450  
                          Unique values in Age : 43    
               Unique values in BusinessTravel : 3     
                   Unique values in Department : 3     
             Unique values in DistanceFromHome : 29    
                    Unique values in Education : 5     
               Unique values in EducationField : 6     
               Unique values in EmployeeNumber : 1450  
      Unique values in EnvironmentSatisfaction : 4     
                       Unique values in Gender : 2     
               Unique values in JobInvolvement : 4     
                     Unique values in JobLevel : 5     
                      Unique values in JobRole : 9     
              Unique values in JobSatisfaction : 4     
                Unique values in MaritalStatus : 3     
                Unique values in MonthlyIncome : 1329  
           Unique values in NumCompaniesWorked : 10    
                       Unique values in Over18 :

### Check Distribution of Numerical Columns

In [12]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['ID']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

Age
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ Age       │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 1450.0    │
│ null_count ┆ 0.0       │
│ mean       ┆ 36.871724 │
│ std        ┆ 9.119033  │
│ min        ┆ 18.0      │
│ 25%        ┆ 30.0      │
│ 50%        ┆ 36.0      │
│ 75%        ┆ 43.0      │
│ max        ┆ 60.0      │
└────────────┴───────────┘ 


DistanceFromHome
shape: (9, 2)
┌────────────┬──────────────────┐
│ statistic  ┆ DistanceFromHome │
│ ---        ┆ ---              │
│ str        ┆ f64              │
╞════════════╪══════════════════╡
│ count      ┆ 1450.0           │
│ null_count ┆ 0.0              │
│ mean       ┆ 9.177241         │
│ std        ┆ 8.085783         │
│ min        ┆ 1.0              │
│ 25%        ┆ 2.0              │
│ 50%        ┆ 7.0              │
│ 75%        ┆ 14.0             │
│ max        ┆ 29.0             │
└────────────┴──────────────────┘ 


Education
shape: (9, 2)
┌─────

### List Unique Values For Certain Features

In [13]:
def list_unique_values_under_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count < threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_under_threshold(df)

Column: ID (1450 unique values)
shape: (1_450,)
Series: 'ID' [i64]
[
	0
	1
	2
	3
	4
	5
	6
	7
	8
	9
	10
	11
	12
	13
	14
	15
	16
	17
	…
	1433
	1434
	1435
	1436
	1437
	1438
	1439
	1440
	1441
	1442
	1443
	1444
	1445
	1446
	1447
	1448
	1449
]
--------------------------------------------------
Column: Age (43 unique values)
shape: (43,)
Series: 'Age' [i64]
[
	18
	19
	20
	21
	22
	23
	24
	25
	26
	27
	28
	29
	30
	31
	32
	33
	34
	35
	…
	44
	45
	46
	47
	48
	49
	50
	51
	52
	53
	54
	55
	56
	57
	58
	59
	60
]
--------------------------------------------------
Column: BusinessTravel (3 unique values)
shape: (3,)
Series: 'BusinessTravel' [str]
[
	"Non-Travel"
	"Travel_Frequently"
	"Travel_Rarely"
]
--------------------------------------------------
Column: Department (3 unique values)
shape: (3,)
Series: 'Department' [str]
[
	"Human Resources"
	"Research & Development"
	"Sales"
]
--------------------------------------------------
Column: DistanceFromHome (29 unique values)
shape: (29,)
Series: 'Distanc